# Transfer Volume

Copies the export volume contents to the target import volume (same metastore).

## Configuration

Read widget parameters: source/target catalogs, schema, and volume names. Derive the export and import volume paths.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("source_schema", "default", "Source schema (export volume location)")
dbutils.widgets.text("target_schema", "default", "Target schema (import volume location)")
dbutils.widgets.text("export_volume", "model_exports", "Export volume name")
dbutils.widgets.text("import_volume", "model_imports", "Import volume name")

SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip()
EXPORT_VOLUME = dbutils.widgets.get("export_volume").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()

if not SOURCE_CATALOG or not TARGET_CATALOG:
    raise ValueError("Set source_catalog and target_catalog")

SRC_PATH = f"/Volumes/{SOURCE_CATALOG}/{SOURCE_SCHEMA}/{EXPORT_VOLUME}"
TGT_PATH = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{IMPORT_VOLUME}"
print(f"Source: {SRC_PATH}")
print(f"Target: {TGT_PATH}")

## Verify export data exists

Checks that the export volume contains a manifest.json before proceeding.

In [ ]:
import os
if not os.path.exists(SRC_PATH) or not os.path.isfile(f"{SRC_PATH}/manifest.json"):
    print("No export data; skipping.")
    dbutils.notebook.exit("SKIP_NO_SOURCE")

## Copy to target import volume

Creates the import volume if needed, clears the target volume, then copies all export files to the target.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}.{IMPORT_VOLUME}")
os.makedirs(TGT_PATH, exist_ok=True)
import shutil, json
if os.path.exists(TGT_PATH):
    for item in os.listdir(TGT_PATH):
        p = os.path.join(TGT_PATH, item)
        if os.path.isdir(p):
            shutil.rmtree(p)
        else:
            os.remove(p)
for entry in sorted(os.listdir(SRC_PATH)):
    src, dst = os.path.join(SRC_PATH, entry), os.path.join(TGT_PATH, entry)
    if os.path.isdir(src):
        shutil.copytree(src, dst)
        print(f"  {entry}/ copied")
    else:
        shutil.copy2(src, dst)
        print(f"  {entry} copied")
with open(f"{SRC_PATH}/manifest.json") as f:
    m = json.load(f)
print(f"TRANSFERRED {m['total_models']} model(s), {m['total_versions']} version(s)")